# Module 1: Jupyter Notebook Basics & REST API
**หลักสูตร:** การพัฒนาเวิร์กโฟลว์ปัญญาประดิษฐ์สำหรับสร้างข้อความและรูปภาพบนเครื่องคอมพิวเตอร์ส่วนบุคคล  
**(Mastering Local AI Stack: Jupyter Notebook × LM Studio × ComfyUI)**

---

## เป้าหมายของบทเรียนนี้
1. แยกแยะหน้าที่ของ `Markdown Cell` กับ `Code Cell` และการเลือก Python Kernel ให้ตรงกับ `.venv` ของโปรเจกต์
2. เขียนโค้ด Python เปิดอ่านไฟล์ข้อความภาษาไทย (`sample_article.txt`)
3. จัดการข้อมูลแบบ **Dictionary / JSON** และนำมาแสดงเป็นตารางด้วย `pandas.DataFrame`
4. ใช้ไลบรารี `requests` ยิงตรวจสอบสถานะของเซิร์ฟเวอร์ **LM Studio** และ **ComfyUI** บนเครื่องตัวเอง

## ส่วนที่ 1: รู้จักกับ Markdown Cell และ Code Cell

พื้นที่ทำงานใน Jupyter Notebook แบ่งออกเป็นกล่องที่เรียกว่า **Cell** โดยเราจะใช้กันอยู่ 2 แบบหลัก:
- **Markdown Cell (กล่องข้อความ):** ใช้จดบันทึก เขียนหัวข้อ หรือใส่คำอธิบายประกอบโค้ด
- **Code Cell (กล่องโค้ด):** ใช้พิมพ์คำสั่งภาษา Python เมื่อกด `Shift + Enter` ตัว **Kernel** จะรันโค้ดในกล่องนั้นแล้วแสดงผลลัพธ์ด้านล่างทันที

In [ ]:
# กด Shift + Enter เพื่อรัน Code Cell แรกและเช็กว่า Python ชี้ไปที่ .venv แล้ว
import sys
import platform

print("=== ข้อมูลสภาพแวดล้อม Python (Environment Info) ===")
print(f"Operating System : {platform.system()} {platform.release()} ({platform.machine()})")
print(f"Python Version   : {sys.version.split()[0]}")
print(f"Executable Path  : {sys.executable}")

## ส่วนที่ 2: การอ่านไฟล์บทความภาษาไทย (`sample_article.txt`)

เวลานำ AI มาช่วยสรุปงาน เรามักต้องอ่านข้อมูลจากไฟล์เอกสารในเครื่องส่งเข้าไปให้โมเดล ในขั้นตอนนี้เราจะลองเปิดอ่านไฟล์ `../assets/sample_article.txt` โดยระบุ `encoding="utf-8"` เพื่อไม่ให้สระและวรรณยุกต์ภาษาไทยเพี้ยน

In [ ]:
from pathlib import Path

# กำหนด Path ไปยังไฟล์บทความตัวอย่าง
article_path = Path("../assets/sample_article.txt")

if article_path.exists():
    article_text = article_path.read_text(encoding="utf-8")
    paragraphs = [p.strip() for p in article_text.split("\n\n") if p.strip()]
    
    print(f"อ่านไฟล์สำเร็จ ความยาวทั้งหมด: {len(article_text):,} ตัวอักษร")
    print(f"จำนวนย่อหน้าทั้งหมด: {len(paragraphs)} ย่อหน้า\n")
    print("--- ตัวอย่างเนื้อหาย่อหน้าที่ 1 ---")
    print(paragraphs[0])
else:
    print(f"ไม่พบไฟล์ที่ {article_path.resolve()}")

## ส่วนที่ 3: การจัดการข้อมูล Dictionary / JSON และ `pandas.DataFrame`

เวลาที่โค้ด Python คุยกับ REST API ของทั้ง LM Studio และ ComfyUI ข้อมูลที่ส่งหากันจะอยู่ในรูปแบบ **JSON (JavaScript Object Notation)** ซึ่งโครงสร้างหน้าตาเหมือนกับ `dict` และ `list` ในภาษา Python

พอได้ข้อมูล JSON กลับมาแล้ว เราสามารถโยนเข้า `pandas.DataFrame` เพื่อดูเป็นตารางหรือเซฟออกเป็นไฟล์ CSV ไปใช้งานต่อได้ทันที

In [ ]:
import json
import pandas as pd

# ข้อมูลสเปกโมเดลที่ใช้ในห้องปฏิบัติการ (VRAM 6GB / Apple Silicon 16GB)
local_ai_models = [
    {
        "component": "LM Studio (LLM)",
        "model_name": "Qwen2.5-3B-Instruct (Q4_K_M)",
        "format": "GGUF",
        "vram_usage_gb": 2.2,
        "recommended_setting": "Context Length <= 2048-4096"
    },
    {
        "component": "LM Studio (LLM)",
        "model_name": "Llama-3.2-3B-Instruct (Q4_K_M)",
        "format": "GGUF",
        "vram_usage_gb": 2.1,
        "recommended_setting": "Context Length <= 2048-4096"
    },
    {
        "component": "ComfyUI (Diffusion)",
        "model_name": "SD 1.5 (v1-5-pruned-emaonly)",
        "format": "Safetensors",
        "vram_usage_gb": 2.0,
        "recommended_setting": "Resolution 512x512, Steps 20"
    }
]

# 1. แปลง Python List/Dict เป็นข้อความ JSON
json_payload = json.dumps(local_ai_models, ensure_ascii=False, indent=2)
print("=== ข้อมูลในรูปแบบ JSON String ===")
print(json_payload)

# 2. แปลงข้อมูลเป็นตาราง Pandas DataFrame
df_models = pd.DataFrame(local_ai_models)
print("\n=== ตารางสรุปการใช้ VRAM (Pandas DataFrame) ===")
display(df_models)

## ส่วนที่ 4: ทดสอบเรียก REST API เช็กสถานะเซิร์ฟเวอร์ด้วย `requests.get()`

ก่อนเริ่มเขียนโค้ดสั่งงานโมเดลในบทถัดไป เราสามารถใช้ `requests.get()` ยิงไปถามเซิร์ฟเวอร์ในเครื่องตัวเอง (Localhost) เพื่อเช็กว่าเปิดโปรแกรมทิ้งไว้พร้อมหรือยัง:
- **LM Studio Local Server:** `http://localhost:1234/v1/models`
- **ComfyUI Server:** `http://127.0.0.1:8188/system_stats`

In [ ]:
import requests

def check_server_status(service_name: str, url: str) -> dict:
    """ฟังก์ชันทดสอบเชื่อมต่อ Local Server ด้วย requests.get()"""
    try:
        response = requests.get(url, timeout=3)
        if response.status_code == 200:
            return {
                "Service": service_name,
                "Endpoint": url,
                "Status": "Online (HTTP 200)",
                "Preview": str(response.json())[:90] + "..."
            }
        return {
            "Service": service_name,
            "Endpoint": url,
            "Status": f"HTTP {response.status_code}",
            "Preview": response.text[:90]
        }
    except requests.exceptions.RequestException as err:
        return {
            "Service": service_name,
            "Endpoint": url,
            "Status": "Offline (ยังไม่ได้กดเปิดเซิร์ฟเวอร์)",
            "Preview": type(err).__name__
        }

# ทดสอบเช็กสถานะทั้ง LM Studio และ ComfyUI
checks = [
    check_server_status("LM Studio API", "http://localhost:1234/v1/models"),
    check_server_status("ComfyUI Server", "http://127.0.0.1:8188/system_stats")
]

df_status = pd.DataFrame(checks)
display(df_status)

---
### ขั้นตอนต่อไป
เมื่อตรวจสอบการอ่านไฟล์และเช็กสถานะ API เบื้องต้นเรียบร้อยแล้ว ให้เปิดใบงานในโฟลเดอร์ `02-lm-studio-api` เพื่อเชื่อมต่อกับ **LM Studio Local Server** และสั่งให้โมเดลขนาด 3B สกัดข้อมูลจากบทความภาษาไทยออกมาเป็น JSON